In [639]:
import pandas as pd
import regex as re
import numpy as np

In [665]:
data = pd.read_csv("data/processed_translation_data_3.csv")

In [666]:
data["author"] = data["author"].str.split(':')
data = data.explode("author")
data["author"] = data["author"].str.strip()

In [667]:
correct_colindex = ["author", "translator", "editor", "fore_afterword_author", 
                    "title", "title_original", "genre", "source_language", "source_literature",
                   "publisher", "publication_location", "publication_year", "physical_description",
                   "series", "publication_type", "target_audience", "edition", "n_pages", "content",
                   "issue", "notes"]

In [668]:
for col in data.columns:
    if col not in correct_colindex:
        print(f"{col} is not in data.columns")

Unnamed: 0 is not in data.columns
author_of_translated_book_under_review_to_be_removed is not in data.columns
id is not in data.columns


In [669]:
data = data.drop(["Unnamed: 0", "id", "author_of_translated_book_under_review_to_be_removed"], axis=1)

In [670]:
data = data.loc[:, correct_colindex]

In [671]:
#data.head(30).to_csv("data/demo-30.csv", index=False)

In [672]:
# Made with the help of ChatGPT (unfortunately)
def split_name_year(series):
    """Split into name and year parts."""
    return series.str.split(r"(\d.*)", n=1, expand=True)[[0,1]]

def clean_text(series):
    return series.str.strip(" \t,")

def split_years(series):
    return series.str.split(r"-", n=1, expand=True)

def clean_year(series):
    return (
        series
        .astype(str)
        .str.replace(r"[^\d\-]", "", regex=True)
        .str.strip()
        .replace("", np.nan)
    )


In [673]:
# Made with the help of ChatGPT (unfortunately)
# --- Build persons dataframe ---
persons = pd.concat([data["author"], data["translator"], data["editor"], data["fore_afterword_author"]]).dropna().drop_duplicates()

persons_df = pd.DataFrame()
persons_df[["name", "year"]] = split_name_year(persons)
persons_df = persons_df.reset_index(drop=True)

# Fix known problematic cases
replacements = {
    "497/496-406/405 eKr: Demokritos u 460- u 370 eKr": "497-406",
    "497/496-406/405 eKr": "497-406",
    "1213(1219)-1292": "1213-1292"
}
persons_df["year"] = persons_df["year"].replace(replacements)

# Split birth/death years using broader separators
persons_df[["birth_year", "death_year"]] = persons_df["year"].str.split(
    r"[\/\-\.—–\(\)]", n=1, expand=True
)

# Clean values
persons_df["birth_year"] = clean_year(persons_df["birth_year"])
persons_df["death_year"] = clean_year(persons_df["death_year"])

# Convert to numeric
persons_df["birth_year"] = pd.to_numeric(persons_df["birth_year"], errors="coerce")
persons_df["death_year"] = pd.to_numeric(persons_df["death_year"], errors="coerce")

# Drop intermediate column
persons_df = persons_df.drop(columns=["year"])

# --- Done ---
print(persons_df.head())

                         name  birth_year  death_year
0                  Zunk, Paul         NaN         NaN
1          Žukovski, Vassili       1783.0      1852.0
2                Žukov, V. V.         NaN         NaN
3          Zshokke, Heinrich       1771.0      1848.0
4  Zschokke, Heinrich Daniel       1771.0      1848.0


In [674]:
persons_df.name = persons_df.name.str.strip()

In [675]:
persons_df.columns

Index(['name', 'birth_year', 'death_year'], dtype='str')

In [676]:
pd.set_option("display.max_columns", 100)
#pd.DataFrame(persons_df.name).head(60)

In [677]:
# Created with the help of Microsoft Copilot
def split_name(name):
    if pd.isna(name):
        return pd.Series([np.nan, np.nan], index=["first_name", "last_name"])

    name = name.strip()

    # Case 1: comma present → "Lastname, Firstname"
    if "," in name:
        last, first = name.split(",", 1)
        last = last.strip()
        first = first.strip()

        # If first part is empty (e.g., "Zeltmatis,")
        if first == "":
            first = np.nan

        return pd.Series([first, last], index=["first_name", "last_name"])

    # Case 2: no comma → split at last space
    if " " in name:
        left, right = name.rsplit(" ", 1)
        left = left.strip()
        right = right.strip()

        # If right part is empty, treat as no split
        if right == "":
            return pd.Series([np.nan, name], index=["first_name", "last_name"])

        return pd.Series([left, right], index=["first_name", "last_name"])

    # Case 3: single word → lastname only
    return pd.Series([np.nan, name], index=["first_name", "last_name"])

In [678]:
persons_df[["first_name", "last_name"]] = persons_df["name"].apply(split_name)

In [679]:
persons_df

,name,birth_year,death_year,first_name,last_name
0,"Zunk, Paul",NaN,NaN,Paul,Zunk
1,"Žukovski, Vassili",1783.0,1852.0,Vassili,Žukovski
2,"Žukov, V. V.",NaN,NaN,V. V.,Žukov
3,"Zshokke, Heinrich",1771.0,1848.0,Heinrich,Zshokke
4,"Zschokke, Heinrich Daniel",1771.0,1848.0,Heinrich Daniel,Zschokke
...,...,...,...,...,...
24883,"Meri, Georg",1900.0,NaN,Georg,Meri
24884,"Riid, Andri",1972.0,NaN,Andri,Riid
24885,"Lewis, Naomi",NaN,NaN,Naomi,Lewis
24886,Mathew Prichard,NaN,NaN,Mathew,Prichard


In [680]:
persons_df = persons_df.drop("name", axis=1)
persons_df = persons_df.loc[:, ["first_name", "last_name", "birth_year", "death_year"]]

In [681]:
persons_df

,first_name,last_name,birth_year,death_year
0,Paul,Zunk,NaN,NaN
1,Vassili,Žukovski,1783.0,1852.0
2,V. V.,Žukov,NaN,NaN
3,Heinrich,Zshokke,1771.0,1848.0
4,Heinrich Daniel,Zschokke,1771.0,1848.0
...,...,...,...,...
24883,Georg,Meri,1900.0,NaN
24884,Andri,Riid,1972.0,NaN
24885,Naomi,Lewis,NaN,NaN
24886,Mathew,Prichard,NaN,NaN


In [682]:
persons_df = persons_df.drop_duplicates()

In [683]:
persons_df.to_csv("data/clean_for_sql/persons.csv", index=False, quoting=False)

In [685]:
data.genre.unique()

<StringArray>
[             'romaanid',                 'jutud',            'luuletused',
             'näidendid',             'ballaadid',                 'luule',
             'aforismid',            'jutustused',               'poeemid',
           'muinasjutud',      'dramatiseeringud',            'humoreskid',
           'proosaluule',             'vanasõnad',                'valmid',
              'novellid',              'teadmata',            'följetonid',
            'anekdoodid',               'eeposed', 'vaimulik ilukirjandus',
                'kirjad',           'biograafiad',              'libretod',
              'päevikud',              'legendid',            'värssjutud',
            'mälestused',               'draamad',            'epigrammid',
                'vested',                'satiir',    'filmistsenaariumid',
        'olukirjeldused',             'komöödiad',             'ulmejutud',
             'pamfletid',                'müüdid',                'saagad'

In [702]:
# Created with the help of Copilot
def make_table(df, column_name, new_name=None):
    if new_name is None:
        new_name = column_name
    return (
        df[[column_name]]
        .dropna()
        .drop_duplicates()
        .sort_values(column_name)
        .reset_index(drop=True)
        .rename(columns={column_name: new_name})
    )

In [722]:
data.source_language = data.source_language.str.strip(": ")
data.source_literature = data.source_literature.str.strip(": ")

In [723]:
genres              = make_table(data, "genre", "est")
literatures         = make_table(data, "source_literature", "est")
publishers          = make_table(data, "publisher", "name")
locations           = make_table(data, "publication_location", "name")
publication_types   = make_table(data, "publication_type", "est")
languages           = make_table(data, "source_language", "est")
series              = make_table(data, "series", "name")
target_audiences    = make_table(data, "target_audience", "est")

In [724]:
genres

,est
0,aabitsad
1,aforismid
2,anekdoodid
3,ballaadid
4,biograafiad
...,...
58,vanasõnad
59,vested
60,värssdraamad
61,värssjutud


In [725]:
genres.to_csv("data/clean_for_sql/genres.csv", index=False, quoting=False)
literatures.to_csv("data/clean_for_sql/literatures.csv", index=False, quoting=False)
publishers.to_csv("data/clean_for_sql/publishers.csv", index=False, quoting=False)
locations.to_csv("data/clean_for_sql/locations.csv", index=False, quoting=False)
publication_types.to_csv("data/clean_for_sql/publication_types.csv", index=False, quoting=False)
languages.to_csv("data/clean_for_sql/languages.csv", index=False, quoting=False)
series.to_csv("data/clean_for_sql/series.csv", index=False, quoting=False)
target_audiences.to_csv("data/clean_for_sql/target_audiences.csv", index=False, quoting=False)

In [714]:
data

,author,translator,editor,fore_afterword_author,title,title_original,genre,source_language,source_literature,publisher,publication_location,publication_year,physical_description,series,publication_type,target_audience,edition,n_pages,content,issue,notes
0,NaN,"Meomuttel, Jüri 1868-1948",NaN,NaN,Natanael. Kulturhistorialine roman. A. Liepe j...,NaN,romaanid,saksa,saksa,NaN,NaN,1896.0,NaN,NaN,Tõlkearvustus,Täiskasvanute ilukirjandus,NaN,"13. veebr., nr 7, lk 161-162",NaN,Olevik,Idna Illiv (pseud.) = Jüri Meomuttel
1,NaN,NaN,NaN,NaN,Talu sulane Tõnis. Ühe Schweitsi jutu järele j...,NaN,jutud,saksa,šveitsi,NaN,NaN,1897.0,NaN,NaN,Tõlkearvustus,Täiskasvanute ilukirjandus,NaN,"28. okt., nr 43, lk 961",NaN,Olevik,NaN
2,"Zunk, Paul","Mällow, J.",NaN,NaN,Miss Portsia,NaN,jutud,NaN,NaN,NaN,NaN,1894.0,NaN,NaN,Perioodikas ilmunud tõlge,Täiskasvanute ilukirjandus,NaN,nr 21-22,NaN,Eesti Postimees : Lisaleht,Väljaandes: Paul Zunk'i uudisjutukese järele J...
3,"Žukovski, Vassili 1783-1852","Leppik, Jaan 1861-1943",NaN,NaN,Kodu,NaN,luuletused,vene,vene,F. Feldt,Viljandi,1889.0,39 lk. 13 cm,NaN,Kogumikus ilmunud tõlge,Täiskasvanute ilukirjandus,NaN,NaN,Ilmunud kogumikus: Eestistatud laulud 2.,NaN,Väljaandes: Wene keelest [autor märkimata]; Ee...
4,"Žukovski, Vassili 1783-1852",G. Kewade,NaN,NaN,Kolm õde,NaN,jutud,saksa,vene,Laakmann,Tartu,1886.0,NaN,NaN,Perioodikas ilmunud tõlge,Täiskasvanute ilukirjandus,NaN,"31. dets., nr. 52, lk. 409",NaN,Meelelahutaja,Väljaandes: Shukowsky järele: G. Kewade
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
78426,"Mérimée, Prosper 1803-1870","Peerna, Rudolf 1864-?","Aavik, Johannes 1880-1973","Aavik, Johannes 1880-1973",Hirmu ja õuduse jutud VII,Die Weissagung; teadmata; Фаталист; Vision de ...,jutud,prantsuse,prantsuse,Reform,Tartu,1917.0,"74, [2] lk","Keelelise uuenduse kirjastik, nr 18",Tõlkeraamat,Täiskasvanute ilukirjandus,NaN,NaN,A. Schnitzler. Ennustus; C. Busse. Üks püha õh...,NaN,A. Schnitzler. Ennustus. Tlk. J. Aavik; C. Bus...
78427,"Busse, Carl 1872-1918","Aavik, Johannes 1880-1973","Aavik, Johannes 1880-1973","Aavik, Johannes 1880-1973",Hirmu ja õuduse jutud VII,Die Weissagung; teadmata; Фаталист; Vision de ...,jutud,prantsuse,prantsuse,Reform,Tartu,1917.0,"74, [2] lk","Keelelise uuenduse kirjastik, nr 18",Tõlkeraamat,Täiskasvanute ilukirjandus,NaN,NaN,A. Schnitzler. Ennustus; C. Busse. Üks püha õh...,NaN,A. Schnitzler. Ennustus. Tlk. J. Aavik; C. Bus...
78428,"Busse, Carl 1872-1918","Reitav, Karl 1897-1961","Aavik, Johannes 1880-1973","Aavik, Johannes 1880-1973",Hirmu ja õuduse jutud VII,Die Weissagung; teadmata; Фаталист; Vision de ...,jutud,prantsuse,prantsuse,Reform,Tartu,1917.0,"74, [2] lk","Keelelise uuenduse kirjastik, nr 18",Tõlkeraamat,Täiskasvanute ilukirjandus,NaN,NaN,A. Schnitzler. Ennustus; C. Busse. Üks püha õh...,NaN,A. Schnitzler. Ennustus. Tlk. J. Aavik; C. Bus...
78429,"Busse, Carl 1872-1918","Moora, Harri 1900-1968","Aavik, Johannes 1880-1973","Aavik, Johannes 1880-1973",Hirmu ja õuduse jutud VII,Die Weissagung; teadmata; Фаталист; Vision de ...,jutud,prantsuse,prantsuse,Reform,Tartu,1917.0,"74, [2] lk","Keelelise uuenduse kirjastik, nr 18",Tõlkeraamat,Täiskasvanute ilukirjandus,NaN,NaN,A. Schnitzler. Ennustus; C. Busse. Üks püha õh...,NaN,A. Schnitzler. Ennustus. Tlk. J. Aavik; C. Bus...


In [715]:
data.to_csv("data/clean_for_sql/translations.csv", index=False, quoting=False)